In [1]:
import re
import json
from google.colab import files

# 1. Load the raw text file
input_filepath = '/content/sa_agniveza-carakasaMhitA-parts-comm.txt'
output_filepath = 'charaka_dataset.json'

with open(input_filepath, 'r', encoding='utf-8') as file:
    content = file.read()

In [2]:
# Strip metadata header
if '# Text' in content:
    text_content = content.split('# Text')[1]
else:
    text_content = content

In [3]:
import re
import json

dataset = []

current_context = "Unknown"
current_shloka = ""
current_commentary = ""
in_commentary = False

# ✅ USE CLEANED TEXT
for line in text_content.splitlines():
    line = line.strip()

    if not line:
        continue

    # CONTEXT
    context_match = re.search(r'(carakasa[ṃm]hitā, [^\n]+)', line)
    if context_match:
        current_context = context_match.group(1).strip()
        continue

    # COMMENTARY START
    if '[{āyurvedadīpikā}' in line:
        in_commentary = True
        part = line.split('[{āyurvedadīpikā}', 1)[1].strip()

        if ']' in part:
            current_commentary = part.split(']', 1)[0].strip()

            if current_shloka and current_commentary:
                dataset.append({
                    "context": current_context,
                    "shloka": current_shloka.strip(),
                    "commentary": current_commentary.strip()
                })

            current_shloka = ""
            current_commentary = ""
            in_commentary = False
        else:
            current_commentary = part

        continue

    # COMMENTARY CONTINUE
    if in_commentary:
        if ']' in line:
            current_commentary += " " + line.split(']', 1)[0].strip()

            if current_shloka and current_commentary:
                dataset.append({
                    "context": current_context,
                    "shloka": current_shloka.strip(),
                    "commentary": current_commentary.strip()
                })

            current_shloka = ""
            current_commentary = ""
            in_commentary = False
        else:
            current_commentary += " " + line

        continue

    # SHLOKA
    current_shloka += " " + line


with open(output_filepath, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)

print("Done! Entries:", len(dataset))

Done! Entries: 295


In [4]:
# 4. Save to JSON file as a proper JSON array
with open(output_filepath, 'w', encoding='utf-8') as outfile:
    # We use json.dump with indent=4 to make the file readable and formatted
    json.dump(dataset, outfile, ensure_ascii=False, indent=4)

print(f"Dataset created with {len(dataset)} items!")

# 5. Automatically trigger download to your local computer
files.download(output_filepath)

Dataset created with 295 items!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import json
from google.colab import files

# 1. Define file paths
input_filepath = '/content/charaka_dataset.json'
output_filepath = 'charaka_tree_dataset.json'

In [6]:
# 2. Load your existing flat JSON dataset
with open(input_filepath, 'r', encoding='utf-8') as file:
    flat_dataset = json.load(file)

# 3. Initialize an empty dictionary to hold the tree structure
tree_dataset = {}

In [7]:
# 4. Iterate through the flat list and group by the "context" key
for item in flat_dataset:
    context_name = item["context"]

    # If this context isn't in our tree yet, create a new list for it
    if context_name not in tree_dataset:
        tree_dataset[context_name] = [] # Fix: Initialize with an empty list

    # Append the shloka and commentary to that specific context's list
    tree_dataset[context_name].append({
        "shloka": item["shloka"],
        "commentary": item["commentary"]
    })

In [8]:
# 5. Save the new hierarchical dataset to a JSON file
with open(output_filepath, 'w', encoding='utf-8') as outfile:
    json.dump(tree_dataset, outfile, ensure_ascii=False, indent=4)

print(f"Successfully grouped the dataset into {len(tree_dataset)} unique contexts!")

# 6. Automatically download the new file to your computer
files.download(output_filepath)

Successfully grouped the dataset into 17 unique contexts!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import json
import re
from google.colab import files

# 1. Define file paths (make sure the input file is uploaded to Colab)
input_filepath = 'charaka_tree_dataset.json'
output_filepath = 'charaka_tree_cleaned.json'

# 2. Load the existing tree-structured dataset
with open(input_filepath, 'r', encoding='utf-8') as file:
    tree_dataset = json.load(file)

# 3. Iterate through the tree and remove citations
for context, entries in tree_dataset.items():
    for entry in entries:
        # Remove citations like "// car_1,1.2" from the shloka
        if "shloka" in entry:
            entry["shloka"] = re.sub(r'//\s*[a-zA-Z0-9_.,]+', '', entry["shloka"]).strip()

        # Remove numerical citations like "// 1" from the commentary
        if "commentary" in entry:
            entry["commentary"] = re.sub(r'//\s*[a-zA-Z0-9_.,]+', '', entry["commentary"]).strip()

# 4. Save the cleaned dataset to a new JSON file
with open(output_filepath, 'w', encoding='utf-8') as outfile:
    json.dump(tree_dataset, outfile, ensure_ascii=False, indent=4)

print("Citations successfully removed from the tree dataset!")

# 5. Automatically download the cleaned file to your computer
files.download(output_filepath)

Citations successfully removed from the tree dataset!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import json
from google.colab import files

# 1. Load your cleaned tree dataset
with open('charaka_tree_cleaned.json', 'r', encoding='utf-8') as f:
    tree_dataset = json.load(f)

flat_training_data = []

# 2. Flatten the tree into conversational prompt strings
for context, entries in tree_dataset.items():
    for entry in entries:
        shloka = entry.get('shloka', '')
        commentary = entry.get('commentary', '')

        # We skip empty entries to ensure data quality
        if shloka and commentary:
            # Format it exactly as the model expects to read it
            prompt = (
                f"<user>Context: {context}\n"
                f"Please explain the following Ayurvedic verse(s):\n"
                f"{shloka}</user>\n"
                f"<assistant>According to Chakrapani's Āyurvedadīpikā commentary: {commentary}</assistant>"
            )

            # Append to our flat list under the "text" key
            flat_training_data.append({"text": prompt})

# 3. Save to JSONL (JSON Lines) format, which is optimal for Hugging Face
output_filepath = 'train_ready_charaka.jsonl'
with open(output_filepath, 'w', encoding='utf-8') as f:
    for item in flat_training_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Ready for fine-tuning! Created {len(flat_training_data)} formatted training examples.")

# 4. Download the final, ready-to-train file
files.download(output_filepath)

Ready for fine-tuning! Created 292 formatted training examples.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>